In [0]:
/*
=============================================================================
PURPOSE: Unified US Treasury Yield Curve Data (Long Format)
=============================================================================
This view consolidates US Treasury yields across multiple maturities (2Y, 5Y, 10Y)
into a single unified table in long/narrow format for time series analysis.

TARGET SCHEMA: thekitchen.t (transformed data layer)
SOURCE VIEWS: t.us_treasury_two_years, t.us_treasury_five_years, t.us_treasury_ten_years
DATE RANGE: 1980-01-01 onwards

BUSINESS VALUE:
- Enables yield curve analysis and visualization (plotting multiple maturities)
- Supports spread calculations (e.g., 10Y-2Y spread for recession indicators)
- Facilitates maturity-based filtering and comparison
- Provides normalized yield data (converted from percentage to decimal)

DATA STRUCTURE:
- One row per date-maturity combination
- Yield normalized to decimal format (e.g., 3.5% → 0.0350)
- MaturityKey provides unique identifier for each maturity-country pair

NOTE: This is TMP1 (temporary/intermediate) version - further transformed
      in us_treasury_rates_tmp2 for CAPM scenario analysis
=============================================================================
*/
CREATE OR REPLACE VIEW t.us_treasury_rates_tmp1 AS
-- ============================================
-- Step 1: Union All Maturities into Long Format
-- ============================================
-- Combines 2Y, 5Y, and 10Y treasury yields into a single dataset
WITH TreasuryYields AS (
  -- 10-Year Treasury Yields
  -- Long-term benchmark rate, key indicator of economic health
  SELECT
    Date,                                        -- Daily observation date
    round((Yield / 100), 4) AS Yield,           -- Convert percentage to decimal (3.5% → 0.0350)
    10 AS MaturityYears,                         -- Maturity in years
    concat("10Y", "-", "US") AS MaturityKey      -- Unique identifier: "10Y-US"
  FROM
    us_treasury_ten_years
  
  UNION ALL
  
  -- 5-Year Treasury Yields
  -- Mid-term benchmark, influences mortgage rates
  SELECT
    Date,
    round((Yield / 100), 4) AS Yield,           -- Convert percentage to decimal
    5 AS MaturityYears,
    concat("5Y", "-", "US") AS MaturityKey       -- Unique identifier: "5Y-US"
  FROM
    us_treasury_five_years
  
  UNION ALL
  
  -- 2-Year Treasury Yields
  -- Short-term rate, sensitive to Fed policy changes
  SELECT
    Date,
    round((Yield / 100), 4) AS Yield,           -- Convert percentage to decimal
    2 AS MaturityYears,
    concat("2Y", "-", "US") AS MaturityKey       -- Unique identifier: "2Y-US"
  FROM
    us_treasury_two_years
)
-- ============================================
-- Step 2: Filter Historical Date Range
-- ============================================
-- Restrict to historical data from 1980 onwards
SELECT
  Date,                  -- Observation date
  Yield,                 -- Normalized yield (decimal format)
  MaturityYears,         -- Years to maturity (2, 5, or 10)
  MaturityKey            -- Composite key for maturity-country combination
FROM
  treasuryyields
WHERE
  Date >= "1980-01-01"   -- Historical cutoff date

/*
=============================================================================
USAGE EXAMPLES:
=============================================================================

-- Example 1: Calculate 10Y-2Y yield spread (recession indicator)
-- Inverted yield curves (negative spread) often precede recessions
WITH yields_wide AS (
  SELECT
    Date,
    MAX(CASE WHEN MaturityYears = 10 THEN Yield END) AS yield_10y,
    MAX(CASE WHEN MaturityYears = 2 THEN Yield END) AS yield_2y
  FROM t.us_treasury_rates_tmp1
  GROUP BY Date
)
SELECT
  Date,
  ROUND(yield_10y, 4) AS yield_10y,
  ROUND(yield_2y, 4) AS yield_2y,
  ROUND((yield_10y - yield_2y), 4) AS spread_10y_2y,
  CASE 
    WHEN (yield_10y - yield_2y) < 0 THEN 'Inverted (Recession Signal)'
    WHEN (yield_10y - yield_2y) < 0.005 THEN 'Flat (Caution)'
    ELSE 'Normal'
  END AS curve_status
FROM yields_wide
WHERE Date >= '2020-01-01'
ORDER BY Date DESC
LIMIT 100;

-- Example 2: Latest yields for all maturities
-- Quick snapshot of current yield curve
SELECT
  MaturityKey,
  MaturityYears,
  ROUND(Yield, 4) AS yield_decimal,
  ROUND(Yield * 100, 2) AS yield_percentage,
  Date
FROM t.us_treasury_rates_tmp1
WHERE Date = (SELECT MAX(Date) FROM t.us_treasury_rates_tmp1)
ORDER BY MaturityYears;

-- Example 3: Yield curve steepness over time
-- Measures the difference between long-term and short-term rates
WITH monthly_avg AS (
  SELECT
    DATE_TRUNC('month', Date) AS month,
    MaturityYears,
    AVG(Yield) AS avg_yield
  FROM t.us_treasury_rates_tmp1
  WHERE Date >= '2020-01-01'
  GROUP BY 1, 2
)
SELECT
  month,
  MAX(CASE WHEN MaturityYears = 2 THEN avg_yield END) AS yield_2y,
  MAX(CASE WHEN MaturityYears = 10 THEN avg_yield END) AS yield_10y,
  (MAX(CASE WHEN MaturityYears = 10 THEN avg_yield END) - 
   MAX(CASE WHEN MaturityYears = 2 THEN avg_yield END)) AS steepness
FROM monthly_avg
GROUP BY month
ORDER BY month DESC;

-- Example 4: Yield curve visualization data
-- Format for charting yield curves on specific dates
SELECT
  Date,
  MaturityYears,
  ROUND(Yield * 100, 2) AS yield_percentage
FROM t.us_treasury_rates_tmp1
WHERE Date IN (
  SELECT MAX(Date) FROM t.us_treasury_rates_tmp1  -- Latest
  UNION
  SELECT MAX(Date) FROM t.us_treasury_rates_tmp1 WHERE Date < DATE_SUB(CURRENT_DATE(), 365)  -- 1 year ago
  UNION
  SELECT MAX(Date) FROM t.us_treasury_rates_tmp1 WHERE Date < DATE_SUB(CURRENT_DATE(), 730)  -- 2 years ago
)
ORDER BY Date DESC, MaturityYears;
=============================================================================
*/